# 13.17 為什麼DPO不用另外教評分員？


我們已經有「同一道題，哪篇回答比較合適」的資料。PPO路線先用它教評分員，再讓策略作答、評分、更新；另一個想法是直接用比較調整策略，省去中間那位獨立評分員。這就是[7.18](https://birdhackor.github.io/tiny-perceptron-vlm/7.18.html)先看過的DPO路線。它仍利用偏好資訊，並不是沒有標準也能自己知道哪篇好。

前置是[13.5的DPO代價轉換](https://birdhackor.github.io/tiny-perceptron-vlm/13.5.html)、[13.12的回答機率](https://birdhackor.github.io/tiny-perceptron-vlm/13.12.html)與[13.14的固定參考](https://birdhackor.github.io/tiny-perceptron-vlm/13.14.html)。對同題較佳與較差回答，DPO比較策略的log機率差，再扣掉固定參考的同一差距，得到margin，也就是相對偏好差距；乘上正係數beta，代價是`−log(sigmoid(beta × margin))`。sigmoid把差距轉成0到1的偏好機率，再取負log，沿用13.5的轉換。減去參考是在問「相對起點，更偏向較佳回答了嗎」，不是強迫每篇回答只增加不減少。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from tiny_perceptron.alignment import dpo_loss

reference = torch.tensor([0.5, 0.5]).log()
for probabilities in [[0.5, 0.5], [0.7, 0.3]]:
    policy = torch.tensor(probabilities).log()
    loss = dpo_loss(policy[:1], policy[1:], reference[:1], reference[1:], beta=1.0)
    print("較佳／較差機率", probabilities, "偏好代價", round(loss.item(), 4))

這份透明例子只有兩張卡，第一張被標為較佳，參考各0.5。目前策略也各0.5時，相對差距0，sigmoid為0.5，代價`−log(0.5)`約0.6931；改成0.7與0.3時，margin是`log(0.7/0.3)`，sigmoid為0.7，代價`−log(0.7)`降為0.3567。這裡beta固定1讓算例容易核對；正式實驗另用0.1，不能直接拿這個小例的代價數字當正式結果。沒有評分網路參與，偏好標籤卻仍決定更新方向。

[DPO原論文第4節公式7](https://arxiv.org/pdf/2305.18290v3)從帶參考約束的偏好目標推導這個方法，不等於「它保證與任何PPO實作得到同一個模型」。真實資料不完美、模型有限、更新配方也不同；兩者都要另外檢查用途。想追完整文字回答的log機率與梯度，接著回[13.3–13.6](https://birdhackor.github.io/tiny-perceptron-vlm/13.3.html)，那條路使用語言模型，與本章新增的有限選卡實驗分開。

圖從同一份示範選卡checkpoint分叉：左路教評分員、取樣與PPO更新；右路直接拿偏好對做DPO。兩路都保留同一個固定起點作reference，最後用相同題目驗收。這不是「先PPO再DPO」的必修流水線。


In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/ppo_dpo_routes.svg")))

正式CPU比較先保存148參數的示範起點，PPO與DPO兩支從逐項相同的權重開始；偏好都來自同一訓練家族。PPO做360次策略更新，DPO也做360次、每批64對，共抽用23,040對。兩者都在最後18題通過12題，且[13.16](https://birdhackor.github.io/tiny-perceptron-vlm/13.16.html)列出的分項相同：數字與求補資訊成功，句子要求仍未學好。但同樣更新360次，不等於同樣計算或資料預算：PPO另有300次評分員更新、360次critic更新、7,680次真實取樣；DPO直接重讀偏好對。

這次PPO段約0.66秒，DPO段約0.34秒；評分員另花約0.27秒，都是本機CPU小網路的段落時間，不是大型LLM效能結論。新增偏好含示範階段沒教的條件，因此也不能說PPO或DPO勝過同資料、同預算的SFT。表格支持「這個固定配方新增了求補資訊的選擇，仍沒完成全部要求」，不支持哪種方法普遍比較好。

當回饋來自真人比較，獎勵模型與PPO可以組成典型RLHF流程；本實驗的標籤則是作者寫定規則由程式套用。最終[19.8](https://birdhackor.github.io/tiny-perceptron-vlm/19.8.html)會把分支與實際成品的選擇說清楚，有限選卡PPO不會被冒稱已回灌成品語言模型。方法可以教在一起，能力與權重來源仍要逐一分清。

練習只把目前策略的第二組機率改成0.3與0.7，先預測它更偏向較差卡，代價升為1.2040，再執行核對。接著指出：即使偏好代價下降，仍缺哪些證據才可發布助理？至少還要實際選擇或生成的任務結果，以及原有能力、格式與誠實等檢查。
